# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [2]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/fewshot_results.jsonl"
MAX_TOKENS  = 8192               # capped from 32768 for tonight's milestone
SEED        = 42                 # fixed for reproducibility
EVAL_N_MCQ  = 10                 # stratified eval subset
EVAL_N_FREE = 10
SUBSET_PATH = "results/eval_subset.json"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}.\n\n"
    "Here are two example solutions in the expected format:\n\n"
    "Example 1:\n"
    "Problem: Compute the derivative f'(x) where f(x) = x^3 - 2x^2 + 5x - 7. f'(x) = [ANS]\n"
    "Solution: Apply the power rule to each term: "
    "d/dx[x^3] = 3x^2, d/dx[-2x^2] = -4x, d/dx[5x] = 5, d/dx[-7] = 0. "
    "Therefore f'(x) = 3x^2 - 4x + 5.\n"
    "\\boxed{3x^2 - 4x + 5}\n\n"
    "Example 2 (multi-part problem with two [ANS] slots):\n"
    "Problem: A rectangle has width w = 4 and length l = 7. "
    "(a) Find the perimeter [ANS]. (b) Find the area [ANS].\n"
    "Solution:\n"
    "(a) Perimeter = 2(w + l) = 2(4 + 7) = 22.\n"
    "(b) Area = w * l = 4 * 7 = 28.\n"
    "\\boxed{22, 28}\n\n"
    "Now solve the user's problem in the same format."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}.\n\n"
    "Here is one example in the expected format:\n\n"
    "Example:\n"
    "Problem: Compute 2 + 2 * 3.\n\n"
    "Options:\n"
    "A. 6\n"
    "B. 8\n"
    "C. 10\n"
    "D. 12\n\n"
    "Solution: By order of operations, 2 + 2*3 = 2 + 6 = 8. The answer is B.\n"
    "\\boxed{B}\n\n"
    "Now answer the user's question in the same format."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} system prompt (first 250 chars) ──")
    print(sys_p[:250], "...\n")


── MCQ system prompt (first 250 chars) ──
You are an expert mathematician. Read the problem and the answer choices below, then select the single best answer. Output ONLY the letter of your chosen option inside \boxed{}, e.g. \boxed{C}.

Here is one example in the expected format:

Example:
P ...

── Free-form system prompt (first 250 chars) ──
You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.

Here are two example solutions in t ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.80,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=16,
    max_num_batched_tokens=8192,
    enforce_eager=True,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-03 19:59:29 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.8, 'max_num_batched_tokens': 8192, 'max_num_seqs': 16, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-03 19:59:30 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 05-03 19:59:30 [model.py:1678] Using max model len 16384
INFO 05-03 19:59:30 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-03 19:59:33 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-03 19:59:33 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-03 19:59:33 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-03 19:59:33 [vllm.py:1025] Cudagraph is disabled under eager mode
INFO 05-03 19:59:33 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant
WARNING 05-03 19:59:35 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usag

(EngineCore pid=59342) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:17<00:34, 17.38s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:34<00:17, 17.22s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:35<00:00,  9.85s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:35<00:00, 11.86s/it]
(EngineCore pid=59342) 


(EngineCore pid=59342) INFO 05-03 20:00:30 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 38.606417 seconds
(EngineCore pid=59342) INFO 05-03 20:00:36 [gpu_worker.py:436] Available KV cache memory: 2.96 GiB
(EngineCore pid=59342) INFO 05-03 20:00:36 [kv_cache_utils.py:1319] GPU KV cache size: 21,536 tokens
(EngineCore pid=59342) INFO 05-03 20:00:36 [kv_cache_utils.py:1324] Maximum concurrency for 16,384 tokens per request: 1.31x
(EngineCore pid=59342) INFO 05-03 20:00:36 [core.py:283] init engine (profile, create kv cache, warmup model) took 6.25 seconds
(EngineCore pid=59342) INFO 05-03 20:00:38 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=59342) WARNING 05-03 20:00:38 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=59342) WARNING 05-03 20:00:38 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that a

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [7]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)
#
# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)
#
# responses = [out.outputs[0].text.strip() for out in outputs]
#
# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


In [8]:
import hashlib
from pathlib import Path

# ── Build (or load) the stratified eval subset ─────────────────────────────
if Path(SUBSET_PATH).exists():
    with open(SUBSET_PATH) as f:
        subset_ids = json.load(f)
    print(f"Loaded existing subset of {len(subset_ids)} IDs from {SUBSET_PATH}")
else:
    import random
    rng = random.Random(SEED)

    def stratify_by_length(items, n):
        items = sorted(items, key=lambda d: len(d["question"]))
        b = len(items) // 3
        buckets = [items[:b], items[b:2*b], items[2*b:]]
        base, rem = divmod(n, 3)
        chosen = []
        for i, bucket in enumerate(buckets):
            k = base + (1 if i < rem else 0)
            chosen.extend(rng.sample(bucket, k))
        return chosen

    mcq_pool  = [d for d in data if d.get("options")]
    free_pool = [d for d in data if not d.get("options")]
    chosen = stratify_by_length(mcq_pool, EVAL_N_MCQ) + stratify_by_length(free_pool, EVAL_N_FREE)
    subset_ids = [d["id"] for d in chosen]

    Path(SUBSET_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(SUBSET_PATH, "w") as f:
        json.dump(subset_ids, f, indent=2)
    print(f"Saved new subset of {len(subset_ids)} IDs to {SUBSET_PATH}")

id_set = set(subset_ids)
subset = [d for d in data if d["id"] in id_set]
n_mcq  = sum(1 for d in subset if d.get("options"))
print(f"Eval subset: {len(subset)} questions ({n_mcq} MCQ, {len(subset)-n_mcq} free-form)")

# ── Cache responses by prompt-hash + seed ─────────────────────────────────
prompt_hash = hashlib.md5(
    (SYSTEM_PROMPT_MATH + "||" + SYSTEM_PROMPT_MCQ).encode()
).hexdigest()[:8]
CACHE_PATH = f"results/cache/{prompt_hash}_seed{SEED}.jsonl"
Path(CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)

cache = {}
if Path(CACHE_PATH).exists():
    with open(CACHE_PATH) as f:
        for line in f:
            e = json.loads(line)
            cache[e["id"]] = e["response"]
print(f"Cache hits: {len(cache)} for prompt hash {prompt_hash}")

to_generate = [item for item in subset if item["id"] not in cache]
print(f"Need to generate: {len(to_generate)} new responses (capped at MAX_TOKENS={MAX_TOKENS})")

# ── Mini-batched generation with per-batch save (crash-safe) ──────────────
# vLLM's generate() returns all results at once, so we split work into
# small batches and save after each one. If the run is killed, only the
# in-flight batch is lost; everything before it survives on disk.
BATCH_SIZE = 5

seeded_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    seed=SEED,
)

for batch_start in range(0, len(to_generate), BATCH_SIZE):
    batch = to_generate[batch_start:batch_start + BATCH_SIZE]
    batch_num = batch_start // BATCH_SIZE + 1
    total_batches = (len(to_generate) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"\n── Batch {batch_num}/{total_batches}: generating {len(batch)} questions ──")

    prompts = []
    for item in batch:
        system, user = build_prompt(item["question"], item.get("options"))
        prompts.append(tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        ))

    outputs = llm.generate(prompts, sampling_params=seeded_params)

    # Save this batch immediately (append-only, crash-safe)
    with open(CACHE_PATH, "a") as f:
        for item, out in zip(batch, outputs):
            resp = out.outputs[0].text.strip()
            cache[item["id"]] = resp
            f.write(json.dumps({"id": item["id"], "response": resp}) + "\n")
    print(f"Saved batch {batch_num}: {len(batch)} responses. Total cached: {len(cache)}/{len(subset)}")

# Align responses with subset order
responses = [cache[item["id"]] for item in subset]
print(f"\nReady: {len(responses)} responses for scoring")

# Preview
for i in range(min(2, len(responses))):
    print(f"\n── Response (id={subset[i]['id']}) ──")
    print(responses[i][:300], "..." if len(responses[i]) > 300 else "")


Loaded existing subset of 20 IDs from results/eval_subset.json
Eval subset: 20 questions (10 MCQ, 10 free-form)
Cache hits: 0 for prompt hash d6308fb1
Need to generate: 20 new responses (capped at MAX_TOKENS=8192)

── Batch 1/4: generating 5 questions ──


Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Saved batch 1: 5 responses. Total cached: 5/20

── Batch 2/4: generating 5 questions ──


Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Saved batch 3: 5 responses. Total cached: 15/20

── Batch 4/4: generating 5 questions ──


Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Saved batch 4: 5 responses. Total cached: 20/20

Ready: 20 responses for scoring

── Response (id=26) ──
Okay, let's try to simplify this expression step by step. The problem is to simplify (125 L^(3/4) P)^(4/3) times P^(-4/3). Hmm, first, I need to remember how exponents work when you have a product raised to a power. Let me recall the rule: (ab)^n = a^n b^n. So maybe I can apply that here.

First, le ...

── Response (id=53) ──
Okay, let's try to figure out this problem. So the question is about the sum of the floor functions of α plus k/n for k from 0 to n-1. Hmm. Let me recall that [x] denotes the floor function, right? So the problem is to find the sum S = [α] + [α + 1/n] + [α + 2/n] + ... + [α + (n-1)/n] where n is a p ...


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [9]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(subset, responses), total=len(subset), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 20/20 [00:00<00:00, 25.24it/s]

Scoring complete. 20 results.


## 8. Summary

Print accuracy broken down by question type.

In [10]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

# Per-length-bucket accuracy (within stratified subset)
sorted_q = sorted(subset, key=lambda d: len(d["question"]))
bsize = len(sorted_q) // 3
length_buckets = {
    "short ": [d["id"] for d in sorted_q[:bsize]],
    "medium": [d["id"] for d in sorted_q[bsize:2*bsize]],
    "long  ": [d["id"] for d in sorted_q[2*bsize:]],
}
print("\nBy question length:")
for label, ids in length_buckets.items():
    s = [r for r in results if r["id"] in set(ids)]
    if s:
        print(f"  {label}: {sum(r['correct'] for r in s):2d} / {len(s):2d}  ({acc(s):.2f}%)")


EVALUATION RESULTS
  MCQ        :    7 /   10  (70.00%)
  Free-form  :    5 /   10  (50.00%)
  Overall    :   12 /   20  (60.00%)

By question length:
  short :  4 /  6  (66.67%)
  medium:  5 /  6  (83.33%)
  long  :  3 /  8  (37.50%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [11]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 20 records to results/fewshot_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!